This file is part of MOFTy.

MOFTy is free software: you can redistribute it and/or modify it under the
terms of the GNU General Public License version 3 as published by the Free
Software Foundation.

MOFTy is distributed in the hope that it will be useful, but WITHOUT ANY
WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS FOR
A PARTICULAR PURPOSE. See the GNU General Public License for more details.

You should have received a copy of the GNU General Public License along with
MOFTy. If not, see http://www.gnu.org/licenses/

Copyright(C) 2026 Maximilian Neumann

In [ ]:
from pathlib import Path
from sklearn.decomposition import PCA

In [ ]:
# --- PCA on concatenated gene expression (HVGs only) + protein (all features) ---
data_dir = Path("../input/input_gbm")
adata_pca = sc.read_h5ad(data_dir / "processed_adata.h5ad")
protein_mask = adata_pca.var["feat_modality"] == "protein"
feature_mask = adata_pca.var["highly_variable"] | protein_mask

X_concat = adata_pca[:, feature_mask].to_df().to_numpy(dtype=np.float64)

# Scale each modality by its overall std (not feature-wise)
feature_mask_idx = np.where(feature_mask.values)[0]
modality_is_gene_exp = adata_pca.var["highly_variable"].values & ~protein_mask.values
modality_is_protein = protein_mask.values
gene_exp_idx = np.where(modality_is_gene_exp)[0]
protein_idx = np.where(modality_is_protein)[0]
gene_exp_cols = np.isin(feature_mask_idx, gene_exp_idx)
protein_cols = np.isin(feature_mask_idx, protein_idx)
gene_exp_std = np.nanstd(X_concat[:, gene_exp_cols])
protein_std = np.nanstd(X_concat[:, protein_cols])
if gene_exp_std > 0:
    X_concat[:, gene_exp_cols] = X_concat[:, gene_exp_cols] / gene_exp_std
if protein_std > 0:
    X_concat[:, protein_cols] = X_concat[:, protein_cols] / protein_std
print(f"Std by modality: gene_exp={gene_exp_std:.4g}, protein={protein_std:.4g}")

print(f"Concatenated matrix for PCA shape: {X_concat.shape}")

# Fit PCA with a larger number of components first
pca_n_comps = 10
pca_concat = PCA(n_components=pca_n_comps, random_state=42)
X_pca_concat = pca_concat.fit_transform(X_concat)

variance_explained_concat = pca_concat.explained_variance_ratio_
print(f"Variance explained by the first 10 PCA components:")
for i, var in enumerate(variance_explained_concat[:10]):
    print(f"PC{i+1}: {(100*var):.1f}%")